In [1]:
"""
Koszt obliczeniowy modeli nienadzorowanych - 3 panele (30 / 20 / 12 cech),
wspolna legenda, limit czasu narysowany jako schodkowy sufit.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, NullLocator, FuncFormatter

# ---------------------------------------------------------------- ustawienia
DATA_DIR = Path(".")

TRANSITIONS = {"AB": "A--B", "AC": "A--C", "BCb": r"B--C$_b$"}
TRANSITION_MARKERS = {"AB": "o", "AC": "s", "BCb": "^"}
FEATURES = [30, 20, 12]

TIME_LIMITS = {5000: 60, 1000: 60, 500: 60, 250: 60,
               100: 300, 50: 600, 10: 1200, 1: 1200}

# granice osi x - uzywane ZAROWNO przez xlim, jak i przez sufit,
# zeby schodkowa linia dochodzila dokladnie do krawedzi ramki
XLIM_LEWO = 5000 * 1.4
XLIM_PRAWO = 1 / 1.4

MODEL_PL = {
    "KMeans": "$k$-średnich",
    "Agglomerative": "Grupowanie aglomeracyjne",
    "Birch": "BIRCH",
    "Spectral": "Grupowanie spektralne",
    "GaussianMixture": "Mieszanina gaussowska",
    "BayesianGMM": "Bayesowska miesz. gaussowska",
    "MeanShift": "Przesunięcie średniej",
    "DBSCAN": "Grupowanie gęstościowe",
}

plt.rcParams.update({
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
    "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.5,
})


def format_timeout(seconds):
    if seconds < 60:
        return f"{seconds:g} s"
    minutes = seconds / 60
    return f"{minutes:g} min" if minutes < 60 else f"{minutes / 60:g} h"


def granice_log(strides):
    """Granice stref na osi log: srodek geometryczny miedzy sasiednimi krokami."""
    s = np.array(sorted(strides, reverse=True), float)
    wewn = np.sqrt(s[:-1] * s[1:])
    return s, np.concatenate([[XLIM_LEWO], wewn, [XLIM_PRAWO]])


def rysuj_pasma(ax, limity, y_min, y_max):
    """Poziome pasma miedzy kolejnymi progami limitu, ciemniejsze ku gorze.

    Pasmo [L_i, L_{i+1}) obejmuje czasy osiagalne wylacznie przy krokach
    o limicie >= L_{i+1}, wiec im wyzej, tym mniej krokow tam siega.
    """
    poziomy = sorted(set(limity.values()))
    krawedzie = poziomy + [y_max]
    for i in range(len(krawedzie) - 1):
        dol, gora = krawedzie[i], krawedzie[i + 1]
        alfa = 0.05 + 0.075 * i           # 0.05, 0.125, 0.20, 0.275
        ax.axhspan(dol, gora, color="0.35", alpha=alfa, lw=0, zorder=0)
        ax.axhline(dol, color="0.55", lw=0.4, alpha=0.5, zorder=1)


def rysuj_sufit(ax, strides, limity):
    """Limit czasu jako schodkowa linia biegnaca przez caly panel."""
    s, gr = granice_log(strides)
    for i, stride in enumerate(s):
        lim = limity[int(stride)]
        ax.plot([gr[i + 1], gr[i]], [lim, lim], color="0.25", lw=1.1,
                ls="-", zorder=2, solid_capstyle="butt")
        if i + 1 < len(s):
            nast = limity[int(s[i + 1])]
            if nast != lim:
                ax.plot([gr[i + 1], gr[i + 1]], [lim, nast], color="0.25",
                        lw=1.1, zorder=2)


# ---------------------------------------------------------------- dane
ramki = []
for kod in TRANSITIONS:
    sciezka = DATA_DIR / "UNSUPERVISED" / kod / f"{kod}_UNS_summary.csv"
    if not sciezka.exists():
        print(f"BRAK: {sciezka}")
        continue
    d = pd.read_csv(sciezka)
    d["transition"] = kod
    ramki.append(d)

uns = pd.concat(ramki, ignore_index=True)
for kol in ("runtime_model", "stride", "n_features"):
    uns[kol] = pd.to_numeric(uns[kol], errors="coerce")
uns = uns.dropna(subset=["model", "runtime_model", "stride", "n_features"])

# kolory przypisane raz, globalnie - te same we wszystkich panelach
modele_all = sorted(uns["model"].unique())
KOLORY = {m: plt.cm.tab10(i % 10) for i, m in enumerate(modele_all)}

# ---------------------------------------------------------------- rysunek
fig, axs = plt.subplots(1, 3, figsize=(7.4, 3.1), sharex=True, sharey=True)

y_min = uns["runtime_model"].min() * 0.5
y_max = max(uns["runtime_model"].max(), max(TIME_LIMITS.values())) * 2.2

narysowane = set()
uzyte_przejscia = set()

for ax, n_cech in zip(axs, FEATURES):
    dane = uns[uns["n_features"] == n_cech]

    rysuj_pasma(ax, TIME_LIMITS, y_min, y_max)
    rysuj_sufit(ax, TIME_LIMITS.keys(), TIME_LIMITS)

    for kod in TRANSITIONS:
        for model in modele_all:
            d = dane[(dane["transition"] == kod) & (dane["model"] == model)]
            if d.empty:
                continue
            d = d.sort_values("stride")
            ax.plot(d["stride"], d["runtime_model"],
                    color=KOLORY[model], marker=TRANSITION_MARKERS[kod],
                    lw=1.1, ms=3.4, alpha=0.9, zorder=3,
                    markeredgecolor="white", markeredgewidth=0.35)
            narysowane.add(model)
            uzyte_przejscia.add(kod)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"{n_cech} cech", loc="left", pad=3)
    ax.set_xlabel("krok szatkowania $s$")

    # os x: wylacznie realnie uzyte kroki szatkowania
    ax.xaxis.set_major_locator(FixedLocator(sorted(TIME_LIMITS)))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v):d}"))
    ax.tick_params(axis="x", labelrotation=45, labelsize=6.5)
    for etykieta in ax.get_xticklabels():
        etykieta.set_horizontalalignment("right")
        etykieta.set_rotation_mode("anchor")

    # siatka: w pionie tylko przy krokach szatkowania, w poziomie pelna
    ax.grid(True, axis="y", which="both", alpha=0.22, lw=0.5)
    ax.grid(True, axis="x", which="major", alpha=0.30, lw=0.5)

axs[0].invert_xaxis()          # sharex -> wystarczy raz
axs[0].set_xlim(XLIM_LEWO, XLIM_PRAWO)
axs[0].set_ylabel("czas grupowania [s]")

# podpisy limitow na prawej osi ostatniego panelu: kazdy siedzi
# dokladnie na wysokosci swojego schodka
poziomy = sorted(set(TIME_LIMITS.values()))
prawa = axs[-1].secondary_yaxis("right")
prawa.set_yticks(poziomy)
prawa.set_yticklabels([format_timeout(p) for p in poziomy])
prawa.tick_params(axis="y", colors="0.45", labelsize=6.5, length=2.5, pad=1.5)
prawa.minorticks_off()
for etykieta in prawa.get_yticklabels():
    etykieta.set_color("0.4")
prawa.spines["right"].set_visible(False)

# ---------------------------------------------------------------- legenda
# budowana WYLACZNIE z tego, co faktycznie narysowano
uchwyty = [Line2D([], [], color=KOLORY[m], marker="o", lw=1.1, ms=3.4,
                  markeredgecolor="white", markeredgewidth=0.35,
                  label=MODEL_PL.get(m, m))
           for m in sorted(narysowane)]
uchwyty += [Line2D([], [], color="0.35", marker=TRANSITION_MARKERS[k],
                   lw=1.1, ms=3.4, label=TRANSITIONS[k])
            for k in TRANSITIONS if k in uzyte_przejscia]
uchwyty += [Line2D([], [], color="0.25", lw=1.1, label="limit czasu")]

fig.tight_layout(rect=[0, 0.13, 1, 1])
fig.legend(handles=uchwyty, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.005), handletextpad=0.4,
           columnspacing=1.2, handlelength=1.6)

for fmt in ("pdf", "png"):
    fig.savefig(f"UNS_runtime_vs_stride.{fmt}",
                bbox_inches="tight", dpi=300 if fmt == "png" else None)

print("modele narysowane:", sorted(narysowane))
print("pominiete w legendzie:", sorted(set(modele_all) - narysowane))

modele narysowane: ['Agglomerative', 'BayesianGMM', 'Birch', 'GaussianMixture', 'KMeans', 'MeanShift', 'Spectral']
pominiete w legendzie: []
